<a href="https://colab.research.google.com/github/english075social/Juego_adivinanza/blob/main/Aguacates.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 🥑 Concurso de Aguacates Gigantes
Analiza de forma interactiva el peso de los aguacates. Calcula estadísticas de centralización, dispersión (incluido el Coeficiente de Variación), forma (asimetría) y genera gráficos representativos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import ipywidgets as widgets
from IPython.display import display, clear_output

# Datos precargados de ejemplo (en gramos)
datos_ejemplo = [350, 420, 450, 480, 500, 510, 550, 590, 610, 680, 720, 850, 310, 490, 530]

# Componentes de la interfaz gráfica
metodo_input = widgets.RadioButtons(
    options=['Usar datos precargados', 'Ingresar datos manualmente'],
    description='Origen de datos:',
    disabled=False,
    style={'description_width': 'initial'}
)

manual_entry = widgets.Text(
    value='350, 420, 450, 480, 500, 510, 550, 590, 610, 680, 720, 850, 310, 490, 530',
    placeholder='Ej: 400, 550, 600, 450',
    description='Pesos manuales (g):',
    disabled=True,
    layout=widgets.Layout(width='70%'),
    style={'description_width': 'initial'}
)

rango_ideal = widgets.IntRangeSlider(
    value=[400, 600],
    min=100,
    max=1200,
    step=10,
    description='Rango ideal de peso (g):',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    layout=widgets.Layout(width='70%'),
    style={'description_width': 'initial'}
)

boton_analizar = widgets.Button(
    description='Ejecutar Análisis 🥑',
    button_style='success',
    tooltip='Haz clic para analizar los datos actuales',
    icon='chart-bar'
)

out = widgets.Output()

def actualizar_campo_entrada(change):
    if change['new'] == 'Ingresar datos manualmente':
        manual_entry.disabled = False
    else:
        manual_entry.disabled = True

metodo_input.observe(actualizar_campo_entrada, names='value')

def realizar_analisis(b):
    with out:
        clear_output(wait=True)

        # Procesamiento de datos de entrada
        if metodo_input.value == 'Usar datos precargados':
            pesos = np.array(datos_ejemplo)
        else:
            try:
                pesos = np.array([float(x.strip()) for x in manual_entry.value.split(',') if x.strip() != ''])
                if len(pesos) == 0:
                    print("❌ Error: Por favor ingresa al menos un valor numérico.")
                    return
            except ValueError:
                print("❌ Error: Formato de entrada no válido. Asegúrate de separar los números únicamente con comas.")
                return

        # Cálculos de estadísticas solicitadas
        media = np.mean(pesos)
        mediana = np.median(pesos)

        try:
            moda_res = stats.mode(pesos, keepdims=True)
            moda = moda_res.mode[0]
        except:
            moda = "N/A"

        desviacion = np.std(pesos, ddof=1) if len(pesos) > 1 else 0.0

        # Coeficiente de variación (Desviación estándar / Media * 100)
        coef_variacion = (desviacion / media) * 100 if media != 0 else 0.0

        # Asimetría (Skewness)
        asimetria = stats.skew(pesos) if len(pesos) > 2 else 0.0

        # Extremos
        pesado = np.max(pesos)
        ligero = np.min(pesos)

        # Conteo dentro del rango ideal
        lim_inf, lim_sup = rango_ideal.value
        dentro_rango = np.sum((pesos >= lim_inf) & (pesos <= lim_sup))

        # Impresión de Resultados en Consola
        print("="*60)
        print("         🥑 RESULTADOS - CONCURSO AGUACATES GIGANTES 🥑")
        print("="*60)
        print(f"• Muestra Total:                  {len(pesos)} aguacates")
        print(f"• Media (Promedio):               {media:.2f} g")
        print(f"• Mediana:                        {mediana:.2f} g")
        print(f"• Moda:                           {moda} g")
        print(f"• Desviación Estándar:            {desviacion:.2f} g")
        print(f"• Coeficiente de Variación (CV):  {coef_variacion:.2f}%")
        print(f"  -> Interpretación: El cultivo es {'homogéneo' if coef_variacion < 15 else 'heterogéneo'} (umbral de ref. 15%)")
        print(f"• Asimetría de la Distribución:   {asimetria:.2f}")
        print(f"• Aguacate más Pesado:            {pesado:.2f} g")
        print(f"• Aguacate más Ligero:            {ligero:.2f} g")
        print(f"• Aguacates en rango ideal ({lim_inf}g - {lim_sup}g): {dentro_rango} ({dentro_rango/len(pesos)*100:.1f}%)")
        print("="*60)

        # Generación de visualizaciones
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))

        # 1. Histograma
        sns.histplot(pesos, kde=True, ax=axes[0], color='darkolivegreen')
        axes[0].axvline(media, color='red', linestyle='--', linewidth=2, label=f'Media: {media:.1f}g')
        axes[0].axvspan(lim_inf, lim_sup, color='yellow', alpha=0.15, label=f'Rango Ideal ({lim_inf}g-{lim_sup}g)')
        axes[0].set_title('Distribución de Pesos de los Aguacates')
        axes[0].set_xlabel('Peso (g)')
        axes[0].set_ylabel('Frecuencia')
        axes[0].legend()

        # 2. Boxplot (Gráfica de caja)
        sns.boxplot(x=pesos, ax=axes[1], color='yellowgreen')
        axes[1].set_title('Identificación de Valores Atípicos (Boxplot)')
        axes[1].set_xlabel('Peso (g)')

        plt.tight_layout()
        plt.show()

        # Exportar datos a reporte.txt
        archivo_reporte = 'reporte_aguacates.txt'
        with open(archivo_reporte, 'w', encoding='utf-8') as f:
            f.write("="*60 + "\n")
            f.write("         🥑 REPORTE: CONCURSO AGUACATES GIGANTES 🥑\n")
            f.write("="*60 + "\n")
            f.write(f"Aguacates analizados: {len(pesos)}\n")
            f.write(f"Media: {media:.2f} g\n")
            f.write(f"Mediana: {mediana:.2f} g\n")
            f.write(f"Moda: {moda} g\n")
            f.write(f"Desviación Estándar: {desviacion:.2f} g\n")
            f.write(f"Coeficiente de Variación: {coef_variacion:.2f}%\n")
            f.write(f"Asimetría del cultivo: {asimetria:.2f}\n")
            f.write(f"Aguacate más pesado: {pesado:.2f} g\n")
            f.write(f"Aguacate más ligero: {ligero:.2f} g\n")
            f.write(f"Aguacates en rango ideal ({lim_inf}g - {lim_sup}g): {dentro_rango} unidades\n")
            f.write("="*60 + "\n")
        print(f"💾 El archivo '{archivo_reporte}' se ha guardado en el entorno local de Colab.")

boton_analizar.on_click(realizar_analisis)

# Presentación de controles
display(widgets.VBox([
    metodo_input,
    manual_entry,
    rango_ideal,
    boton_analizar,
    out
]))